In [10]:
# %% [markdown]
# # Assignment 1 - Part 3.3: Sequence to Histogram
#
# This notebook-style script implements the third autoregressive model:
# **Sequence to Histogram**.
#
# Requirement target:
# - Use over 512 images for learning.
# - Generated sequences should match training sequences over 99% of the time.
#
# Main idea:
# - Convert each binary latent code from the trained autoencoder into a 48-token sequence.
# - For each prefix, train the model to output a histogram:
#   `P(next token = 0 | prefix)` and `P(next token = 1 | prefix)`.
# - At generation time, decode each prefix greedily from the predicted histogram.


In [11]:
# %%
import argparse
import importlib.util
import sys
from pathlib import Path
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from tqdm import tqdm

In [12]:
# %% [markdown]
# ## 1. Load AutoEncoder Class
#
# Update `AE_MODULE_PATH` and `AE_WEIGHTS_PATH` only if your Kaggle input dataset name changes.

# %%
AE_MODULE_PATH = Path("/kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_model.py")
AE_WEIGHTS_PATH = "/kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_best.pth"

if not AE_MODULE_PATH.exists():
    AE_MODULE_PATH = Path("/kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_model.py")
    AE_WEIGHTS_PATH = "/kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_best.pth"

spec = importlib.util.spec_from_file_location("autoencoder_model", AE_MODULE_PATH)
autoencoder_model = importlib.util.module_from_spec(spec)
sys.modules["autoencoder_model"] = autoencoder_model
spec.loader.exec_module(autoencoder_model)
AutoEncoder = autoencoder_model.AutoEncoder



In [13]:
# %% [markdown]
# ## 2. Dataset: Prefix to Histogram

class PrefixHistogramDataset(Dataset):
    def __init__(self, autoencoder, images, device, seq_len=48):
        self.device = device
        self.seq_len = seq_len
        autoencoder.eval()
 
        with torch.no_grad():
            imgs_normalized = images.to(device) * 2 - 1
            _, latents, _ = autoencoder(imgs_normalized)
 
        self.latents = latents.view(latents.size(0), -1)
        self.sequences = ((self.latents + 1) / 2).long().cpu()
 
        # CHANGE HERE -> count next-token occurrences for every unique prefix
        counts = {}
        for seq in self.sequences.tolist():
            for i in range(seq_len):
                c = counts.setdefault(tuple(seq[:i]), [0, 0])
                c[seq[i]] += 1
 
        inputs, lengths, targets = [], [], []
        for prefix, (c0, c1) in counts.items():
            padded = torch.zeros(seq_len, dtype=torch.long)
            if len(prefix) > 0:
                padded[: len(prefix)] = torch.tensor(prefix, dtype=torch.long)
            inputs.append(padded)
            lengths.append(len(prefix))
            # CHANGE HERE -> true histogram (e.g. [0.5, 0.5]) instead of a one-hot target
            targets.append(torch.tensor([c0, c1], dtype=torch.float32) / (c0 + c1))
 
        self.inputs = torch.stack(inputs)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.targets = torch.stack(targets)
 
        n_ambiguous = sum(1 for a, b in counts.values() if a > 0 and b > 0)
        print(f"Training images: {len(self.sequences)}")
        print(f"Sequence length: {seq_len}")
        print(f"Unique prefixes: {len(self.inputs)} | ambiguous (mixed histogram): {n_ambiguous}")
 
    def __len__(self):
        return len(self.inputs)
 
    def __getitem__(self, index):
        return self.inputs[index], self.lengths[index], self.targets[index]
 


In [14]:
# %% [markdown]
# ## 3. Sequence to Histogram Model
#
# The model reads the prefix and outputs a 2-bin histogram for the next token.
# Output bins are logits for:
# - class 0: next token is 0
# - class 1: next token is 1

# %%
class Seq2HistogramTransformer(nn.Module):
    def __init__(self, seq_len=48, vocab_size=2, d_model=192, nhead=8, num_layers=4, dropout=0.0):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model

        self.token_embedding = nn.Embedding(vocab_size + 1, d_model)
        self.length_embedding = nn.Embedding(seq_len + 1, d_model)
        self.position_embedding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.query_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2),
        )

    def forward(self, tokens, lengths):
        batch_size = tokens.size(0)
        positions = torch.arange(self.seq_len, device=tokens.device).unsqueeze(0)

        visible_mask = positions < lengths.unsqueeze(1)
        masked_tokens = tokens.clone()
        masked_tokens[~visible_mask] = 2

        x = self.token_embedding(masked_tokens)
        x = x + self.position_embedding
        x = x + self.length_embedding(lengths).unsqueeze(1)

        query = self.query_token.expand(batch_size, -1, -1)
        x = torch.cat([query, x], dim=1)

        pad_mask = torch.cat(
            [
                torch.zeros(batch_size, 1, dtype=torch.bool, device=tokens.device),
                ~visible_mask,
            ],
            dim=1,
        )

        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.output_head(h[:, 0])


In [15]:

# %% [markdown]
# ## 4. Training

# %%
def train_seq2histogram(model, loader, device, epochs=400, lr=5e-4, warmup_epochs=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))

    history = {"loss": [], "acc": [], "det_acc": []}  # CHANGE HERE -> added det_acc

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0
        det_correct = 0  # CHANGE HERE -> deterministic-prefix counters
        det_total = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
        for tokens, lengths, hist in pbar:
            tokens = tokens.to(device)
            lengths = lengths.to(device)
            hist = hist.to(device)
            labels = hist.argmax(dim=1)
 
            logits = model(tokens, lengths)
            log_probs = F.log_softmax(logits, dim=1)
            loss = -(hist * log_probs).sum(dim=1).mean()  # works as-is with soft histogram targets
 
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
 
            total_loss += loss.item() * tokens.size(0)
            pred = logits.argmax(dim=1)
            total_correct += (pred == labels).sum().item()
            total_samples += tokens.size(0)
 
            # CHANGE HERE -> accuracy only on prefixes with a single possible next token.
            # THIS is the number that must reach 1.0 for 99% memorization.
            det = hist.max(dim=1).values == 1.0
            det_correct += (pred[det] == labels[det]).sum().item()
            det_total += det.sum().item()
 
            pbar.set_postfix(loss=total_loss / total_samples, det_acc=det_correct / max(1, det_total))
 
        scheduler.step()
 
        avg_loss = total_loss / total_samples
        avg_acc = total_correct / total_samples
        avg_det_acc = det_correct / max(1, det_total)  # CHANGE HERE
        history["loss"].append(avg_loss)
        history["acc"].append(avg_acc)
        history["det_acc"].append(avg_det_acc)  # CHANGE HERE
 
        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | Loss: {avg_loss:.5f} | "
                f"Acc: {avg_acc:.4f} | Deterministic acc: {avg_det_acc:.5f} | "  # CHANGE HERE
                f"LR: {scheduler.get_last_lr()[0]:.6f}"
            )
 
    return history
 

In [16]:
# %% [markdown]
# ## 5. Generation
#
# To make the model a memorization model, generation is greedy:
# at each step choose the most probable token from the predicted histogram.

# %%
@torch.no_grad()
def generate_sequences(model, device, num_samples=1024, seq_len=48,start_mode="empty", train_sequences=None):
    model.eval()
    generated = torch.zeros(num_samples, seq_len, dtype=torch.long, device=device)

    if start_mode == "train_prefixes" and train_sequences is not None:
        train_sequences = train_sequences.to(device).long()
        repeats = (num_samples + train_sequences.size(0) - 1) // train_sequences.size(0)
        generated[:, 0] = train_sequences.repeat(repeats, 1)[:num_samples, 0]
        start_index = 1
    else:
        start_index = 0

    for i in range(start_index, seq_len):
        lengths = torch.full((num_samples,), i, dtype=torch.long, device=device)
        logits = model(generated, lengths)
        probs = torch.softmax(logits, dim=1)
        generated[:, i] = torch.multinomial(probs, 1).squeeze(1)

    return generated.float()


def decode_sequences(autoencoder, token_sequences, device):
    autoencoder.eval()
    latent_sequences = token_sequences.to(device) * 2 - 1
    latent_sequences = latent_sequences.view(-1, 3, 4, 4)

    with torch.no_grad():
        imgs = autoencoder.decoder(latent_sequences)

    return imgs

In [17]:
# %% [markdown]
# ## 6. Memorization and Visualization

# %%
def check_memorization(generated_seqs, train_seqs):
    generated_seqs = generated_seqs.cpu()
    train_seqs = train_seqs.float().cpu()
    diffs = (generated_seqs.unsqueeze(1) - train_seqs.unsqueeze(0)).abs().sum(dim=-1)
    matches = (diffs == 0).any(dim=1).float()
    return matches.mean().item(), matches


def find_closest_images(generated_imgs, train_imgs):
    closest = []
    for gen_img in generated_imgs:
        dists = (gen_img - train_imgs).abs().reshape(train_imgs.size(0), -1).sum(dim=1)
        closest_idx = dists.argmin()
        closest.append(train_imgs[closest_idx])
    return torch.stack(closest)


def visualize_results(original_imgs, reconstructed_imgs, generated_imgs, closest_imgs, save_path):
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))

    grid = make_grid(original_imgs[:64], nrow=8, normalize=True, value_range=(-1, 1))
    axes[0].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Original MNIST images")
    axes[0].axis("off")

    grid = make_grid(reconstructed_imgs[:64], nrow=8, normalize=True, value_range=(-1, 1))
    axes[1].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[1].set_title("Reconstructed by AutoEncoder")
    axes[1].axis("off")

    grid = make_grid(generated_imgs[:64], nrow=8, normalize=True, value_range=(-1, 1))
    axes[2].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[2].set_title("Generated by Sequence to Histogram model")
    axes[2].axis("off")

    grid = make_grid(closest_imgs[:64], nrow=8, normalize=True, value_range=(-1, 1))
    axes[3].imshow(grid.permute(1, 2, 0).cpu().numpy())
    axes[3].set_title("Closest training reconstructions")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved visualization -> {save_path}")
    plt.close()


In [18]:
# %% [markdown]
# ## 7. Main Experiment

# %%
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=400)
    parser.add_argument("--num_samples", type=int, default=1025)
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--d_model", type=int, default=192)
    parser.add_argument("--nhead", type=int, default=8)
    parser.add_argument("--num_layers", type=int, default=4)
    parser.add_argument("--num_generated", type=int, default=512)
    args, unknown = parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"AutoEncoder module: {AE_MODULE_PATH}")
    print(f"AutoEncoder weights: {AE_WEIGHTS_PATH}")

    ae = AutoEncoder(use_binary=True).to(device)
    ae.load_state_dict(torch.load(AE_WEIGHTS_PATH, map_location=device))
    ae.eval()
    print("OK - Loaded autoencoder_best.pth")

    tfm = transforms.Compose([transforms.Resize(32), transforms.ToTensor()])
    mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
    images = torch.stack([img for img, _ in list(mnist)[: args.num_samples]])

    dataset = PrefixHistogramDataset(ae, images, device, seq_len=48)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=0)

    model = Seq2HistogramTransformer(
        seq_len=48,
        d_model=args.d_model,
        nhead=args.nhead,
        num_layers=args.num_layers,
        dropout=0.0,
    ).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    history = train_seq2histogram(model, loader, device, epochs=args.epochs, lr=args.lr)

    torch.save(model.state_dict(), "seq2histogram_transformer.pth")
    print("OK - Saved model -> seq2histogram_transformer.pth")

    generated_seqs = generate_sequences(
        model,
        device,
        num_samples=args.num_generated,
        seq_len=48,
        start_mode="empty",
        train_sequences=dataset.sequences,
    )

    memo_ratio, matches = check_memorization(generated_seqs, dataset.sequences)
 
    # CHANGE HERE -> sanity check: should be a large number, not 2
    n_unique = generated_seqs.unique(dim=0).size(0)

    generated_imgs = decode_sequences(ae, generated_seqs, device)
    with torch.no_grad():
        imgs_normalized = images[: args.num_generated].to(device) * 2 - 1
        reconstructed_imgs, _, _ = ae(imgs_normalized)
        all_train_imgs = images.to(device) * 2 - 1
        all_train_reconstructed, _, _ = ae(all_train_imgs)

    closest_imgs = find_closest_images(generated_imgs, all_train_reconstructed)

    visualize_results(
        imgs_normalized,
        reconstructed_imgs,
        generated_imgs,
        closest_imgs,
        save_path="q3_seq2histogram_results.png",
    )
    save_image((generated_imgs + 1) / 2, "q3_seq2histogram_generated.png", nrow=8)

    print("=" * 60)
    print("SUMMARY (Sequence to Histogram)")
    print("=" * 60)
    print(f"Training samples: {args.num_samples}")
    print(f"Generated samples: {args.num_generated}")
    print(f"Unique generated sequences: {n_unique}")  # CHANGE HERE
    print(f"Epochs: {args.epochs}")
    print(f"Final loss: {history['loss'][-1]:.5f}")
    print(f"Final overall histogram accuracy: {history['acc'][-1]:.4f} (cannot reach 1.0 by design)")
    print(f"Final deterministic-prefix accuracy: {history['det_acc'][-1]:.5f}")  # CHANGE HERE
    print(f"Memorization: {memo_ratio * 100:.2f}%")
    print(f"Exact matches: {int(matches.sum().item())}/{args.num_generated}")
    if args.num_samples > 512 and memo_ratio >= 0.99:  # CHANGE HERE -> ">" because the assignment says "over 512":
        print("SUCCESS: target reached, over 512 images and >=99% memorization.")
    else:
        print("If below target, try --epochs 600 --d_model 256 --num_layers 6")
    print("=" * 60)


if __name__ == "__main__":
    main()


Using device: cuda
AutoEncoder module: /kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_model.py
AutoEncoder weights: /kaggle/input/datasets/saifmukadam10/helper-data-python-scripts-and-model/autoencoder_best.pth
OK - Loaded autoencoder_best.pth


/tmp/ipykernel_49/1392904426.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Training images: 1025
Sequence length: 48
Unique prefixes: 32068 | ambiguous (mixed histogram): 1020
Model parameters: 1,836,674


Epoch 001 | Loss: 0.58736 | Acc: 0.6740 | Deterministic acc: 0.67756 | LR: 0.000500


Epoch 010 | Loss: 0.53422 | Acc: 0.7201 | Deterministic acc: 0.72427 | LR: 0.000499


...

Epoch 340 | Loss: 0.02001 | Acc: 0.9934 | Deterministic acc: 0.99974 | LR: 0.000027


Epoch 350 | Loss: 0.01981 | Acc: 0.9935 | Deterministic acc: 0.99977 | LR: 0.000019


Epoch 360 | Loss: 0.01970 | Acc: 0.9934 | Deterministic acc: 0.99987 | LR: 0.000012


Epoch 370/400:  53%|█████▎    | 132/251 [00:05<00:04, 24.82it/s, det_acc=1, loss=0.0202]   IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)



Epoch 400 | Loss: 0.01957 | Acc: 0.9942 | Deterministic acc: 0.99994 | LR: 0.000000
OK - Saved model -> seq2histogram_transformer.pth
Saved visualization -> q3_seq2histogram_results.png
SUMMARY (Sequence to Histogram)
Training samples: 1025
Generated samples: 512
Unique generated sequences: 411
Epochs: 400
Final loss: 0.01957
Final overall histogram accuracy: 0.9942 (cannot reach 1.0 by design)
Final deterministic-prefix accuracy: 0.99994
Memorization: 99.41%
Exact matches: 509/512
SUCCESS: target reached, over 512 images and >=99% memorization.


In [ ]:
Epoch 300 | Loss: 0.02150 | Acc: 0.9925 | Deterministic acc: 0.99942 | LR: 0.000073
                                                                                            
Epoch 310 | Loss: 0.02086 | Acc: 0.9932 | Deterministic acc: 0.99968 | LR: 0.000060
                                                                                            
Epoch 340 | Loss: 0.02001 | Acc: 0.9934 | Deterministic acc: 0.99974 | LR: 0.000027
                                                                                           
Epoch 350 | Loss: 0.01981 | Acc: 0.9935 | Deterministic acc: 0.99977 | LR: 0.000019
                                                                                           
Epoch 360 | Loss: 0.01970 | Acc: 0.9934 | Deterministic acc: 0.99987 | LR: 0.000012
Epoch 370/400:  53%|█████▎    | 132/251 [00:05<00:04, 24.82it/s, det_acc=1, loss=0.0202]   IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)

                                                                                           
Epoch 400 | Loss: 0.01957 | Acc: 0.9942 | Deterministic acc: 0.99994 | LR: 0.000000
OK - Saved model -> seq2histogram_transformer.pth
Saved visualization -> q3_seq2histogram_results.png
============================================================
SUMMARY (Sequence to Histogram)
============================================================
Training samples: 1025
Generated samples: 512
Unique generated sequences: 411
Epochs: 400
Final loss: 0.01957
Final overall histogram accuracy: 0.9942 (cannot reach 1.0 by design)
Final deterministic-prefix accuracy: 0.99994
Memorization: 99.41%
Exact matches: 509/512
SUCCESS: target reached, over 512 images and >=99% memorization.
============================================================